# One-vs-all CatBoost classifiers per class on lemmatized text with keyword features

# Import and Installing Library

In [ ]:
DATA_DIR = "../data"

In [ ]:
!pip install -q CatBoost
!pip install -q datasets

In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback
)

import tensorflow as tf
from catboost import CatBoostClassifier, Pool

from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, recall_score, precision_score, \
    balanced_accuracy_score
from datasets import Dataset, DatasetDict

In [ ]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# Loading and Cleaning Data

In [ ]:
train_df = pd.read_csv(f'{DATA_DIR}/bdc_train.csv', sep=";")
# ft_embed = pd.read_pickle(f'{DATA_DIR}/ft_embed.pkl')
# test_df = pd.read_csv(f'{DATA_DIR}/test.csv', sep=";")
train_df.head()

                                                text             label
0  Kunjungan Prabowo ini untuk meresmikan dan men...  Sumber Daya Alam
1  RT Anies dapat tepuk tangan meriah saat jadi R...           Politik
2  @CIqXqwGAT04tMtx4OCATxjoVq7vv/Y8HeYaIOgMFg8Y= ...         Demografi
3  RT @L3R8XFBw3WGbxRPSj0/0hHZTbqVGX7qtfwRg9zmhK7...           Politik
4  Anies Baswedan Harap ASN termasuk TNI dan Polr...           Politik

In [ ]:
# train_df['ft_embed'] = ft_embed['ft_embed']

In [ ]:
import re
def preprocess_hashtag(hashtag):
    hashtag = hashtag.replace("M3nang", "Menang").replace("L3bih", "Lebih")
    hashtag = re.sub(r'(?i)g4njar', 'Ganjar', hashtag)
    hashtag = hashtag.lstrip('#')  # Remove the '#' symbol
    words = re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?![a-z])|\d+', hashtag)
    return ' '.join(words).lower()  # Convert to lowercase and join with spaces

def replace_hashtags(text):
    # Find all hashtags in the text
    hashtags = re.findall(r'#\w+', text)

    # Process each hashtag and replace it in the text
    for hashtag in hashtags:
        processed = preprocess_hashtag(hashtag)
        text = text.replace(hashtag, processed)

    return text

In [ ]:
def remove_noise(text):
    text = str(text)
    # remove mentions
    text = re.sub(r'@[\S+]+', '', text)
    # remove hashtags
    # text = re.sub(r'#\w+', '', text)
    # remove RT
    text = re.sub(r'RT[\s]+', '', text)
    # remove [RE ...]
    text = re.sub(r'\[RE\s+(\w+)\]', r'\1', text)
    # remove urls
    text = re.sub(r'https?:\/\/\S+', '', text)
    # remove emojis
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # fix &amp;
    text = re.sub(r'&amp;', 'dan', text)
    # fix w/
    text = re.sub(r'w/', 'with', text)
    # fix 3
    text = re.sub(r'(?<=\w)3(?=\w)', 'e', text)
    # remove any non alphanumeric characters
    text = re.sub(r'[^A-Za-z0-9 ]', ' ', text)
    # fix unspaced words
    new_text = []
    for word in text.split():
        # weird word
        if word in ["BEiqnH5xQr", "JN18A"]:
            continue
        # fix shorten names
        elif word == "JK":
            new_text.append("jusuf kalla")
        elif word.lower() == "jkw":
            new_text.append("jokowi")
        elif word.lower() in ["bowo", "ayahbowo"]:
            new_text.append("prabowo")
        elif word.lower() in ["ayahbowogbran", "pbowogbran", "masbowogbran"]:
            new_text.append("pabrowo gibran")
        elif word == "MasGBRAN":
            new_text.append("gibran")
        elif word == "SBY":
            new_text.append("susilo bambang yudhoyono")
        elif word == "AHY":
            new_text.append("agnes haryani")
        # keep as is
        elif re.fullmatch(r'[A-Z]+', word) or word.lower() in ["s1", "s2", "s3"]:
            new_text.append(word)
        else:
            # fix unspaced words
            new_text.append(re.sub(r'(?<=[a-z])(?=[A-Z])|(?<=[A-Za-z])(?=[0-9])|(?<=[0-9])(?=[A-Za-z])|(?<=[A-Z])(?=[A-Z][a-z])', ' ', word))
    text = ' '.join(new_text)
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    # remove whitespace leading & trailing
    text = text.strip()
    return text

def normalize_word(text):
    map = {
        "sj": "saja",
        "yg": "yang",
        "dgn": "dengan",
        "dg": "dengan",
        "dpt": "dapat",
        "dptkan": "dapatkan",
        "hnya": "hanya",
        "skrng": "sekarang",
        "uu": "undang undang",
        "asn": "aparatur sipil negara",
        "ordal": "orang dalam",
        "mrk": "mereka",
        "rmh": "rumah",
        "kpk": "komisi pemberantasan korupsi",
        "blg": "bilang",
        "tuch": "tuh",
        "umkm": "usaha mikro kecil menengah",
        "jis": "jakarta international stadium",
        "jkt": "jakarta",
        "spt": "seperti",
        "tps": "tempat pemungutan suara",
        "alkes": "alat kesehatan",
        "bowo": "prabowo",
        "maap": "maaf",
        "ga" : "tidak",
        "nggak" : "tidak"
    }

    text = text.replace("UU ITE", "Undang Undang Informasi dan Transaksi Elektronik")
    text = text.replace("kitamasbowo dangbran", "kita pilih prabowo dan gibran")
    text = text.replace("PDI P", "PDIP")
    text = text.replace("PDI Perjuangan", "PDIP")

    party_names = [
        "pkb",
        "gerindra",
        "pdi",
        "pdip",
        "golkar",
        "nasdem",
        "gelora",
        "pks",
        "pkn",
        "hanura",
        "pan",
        "pbb",
        "demokrat",
        "psi",
        "perindo",
        "ppp",
        "ummat",
        "pna",
        "gabthat",
        "pda",
        "sira"
    ]

    stop_words = [
        "dekade08",
        "tempodotco",
        "md"
    ]

    new_text = []
    for word in text.split():
        if word.lower() in map:
            new_text.append(map[word.lower()])
        elif word.lower() in party_names:
            new_text.append("Partai Politik")
        elif word.lower() in stop_words:
            continue
        else:
            new_text.append(word)

    text = ' '.join(new_text)
    return text.lower()

def drop_duplicated(df):
    print("Df size:", df.shape)
    df.drop(df[df['text'].str.len() == 0].index, inplace=True)
    print("After dropping empty text rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text', 'label'], keep='first', inplace=True)
    print("After dropping literal duplicated rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text'], keep=False, inplace=True)
    print("After dropping text duplicated rows, df size: ", df.shape)
    return df

In [ ]:
!pip install -q nlp_id

In [ ]:
from nlp_id.lemmatizer import Lemmatizer
lemmatizer = Lemmatizer()
lemmatizer.lemmatize('Saya sedang mencoba')

'saya sedang coba'

In [ ]:
# train_df['text'] = train_df['text'].apply(replace_hashtags)
train_df['text'] = train_df['text'].apply(remove_noise)
train_df['text'] = train_df['text'].apply(normalize_word)
train_df['text_2'] = train_df['text'].apply(lambda x: lemmatizer.lemmatize(x))
train_df = drop_duplicated(train_df)
train_df.head()

Df size: (5000, 3)
After dropping empty text rows, df size:  (4991, 3)
After dropping literal duplicated rows, df size:  (4352, 3)
After dropping text duplicated rows, df size:  (4239, 3)


                                                text             label  \
0  kunjungan prabowo ini untuk meresmikan dan men...  Sumber Daya Alam   
1  anies dapat tepuk tangan meriah saat jadi rekt...           Politik   
2  emng bener sih pendukung 01 ada yang goblok be...         Demografi   
3  sewaktu anies bersikap kritis ke kinerja pak p...           Politik   
4  anies baswedan harap aparatur sipil negara ter...           Politik   

                                              text_2  
0  kunjung prabowo ini untuk resmi dan serah proy...  
1  anies dapat tepuk tangan riah saat jadi rektor...  
2  emng bener sih dukung 01 ada yang goblok begit...  
3  waktu anies sikap kritis ke kerja pak prabowo ...  
4  anies baswedan harap aparatur sipil negara mas...  

In [ ]:
train_df.head()

                                                text             label  \
0  kunjungan prabowo ini untuk meresmikan dan men...  Sumber Daya Alam   
1  anies dapat tepuk tangan meriah saat jadi rekt...           Politik   
2  emng bener sih pendukung 01 ada yang goblok be...         Demografi   
3  sewaktu anies bersikap kritis ke kinerja pak p...           Politik   
4  anies baswedan harap aparatur sipil negara ter...           Politik   

                                              text_2  
0  kunjung prabowo ini untuk resmi dan serah proy...  
1  anies dapat tepuk tangan riah saat jadi rektor...  
2  emng bener sih dukung 01 ada yang goblok begit...  
3  waktu anies sikap kritis ke kerja pak prabowo ...  
4  anies baswedan harap aparatur sipil negara mas...  

In [ ]:
train_df[train_df['label'] == 'Geografi'].to_csv(f"{DATA_DIR}/geo_clean.csv")

In [ ]:
train_df.to_csv(f"{DATA_DIR}/train_cleaned_v2.csv", index=False)

In [ ]:
train_df.label.value_counts()

label
Politik                    2867
Sosial Budaya               384
Ekonomi                     264
Pertahanan dan Keamanan     259
Ideologi                    255
Sumber Daya Alam            133
Demografi                    55
Geografi                     19
Name: count, dtype: int64

In [ ]:
test_df['Text'] = test_df['Text'].apply(remove_noise)
test_df['Text'] = test_df['Text'].apply(normalize_word)
test_df.head()

,IDText,Text
0,TXT0001,Lu mau org 2 pro demokrasi di negara ini bisa ...
1,TXT0002,Prabowo ditanya soal hutang luar negeri dia me...
2,TXT0003,kiki daliyo Ganjar Pranowo itulah beliau sosok...
3,TXT0004,Prabowo Gibran yang bisa melakukan itu semua d...
4,TXT0005,Lah justru yang gak nyambung junjungan elu aom...


In [ ]:
train_df[train_df['text'].isin(test_df['Text'])].drop_duplicates(subset=['text'])

,text,label
64,Anies itu prestasinya di Jakarta banyak sekali...,Politik
68,Isu Palestina tidak penting bagi Prabowo tidak...,Politik
87,Palestina di tindas karena gk punya kekuatan m...,Pertahanan dan Keamanan
160,Partai Politik Tanggapi Jokowi Prabowo Makan M...,Politik
270,nahh itu dia laeku ga bisa dibuka semua data i...,Politik
...,...,...
4869,Jadi ikut sedih juga Pak Prabowo memang sering...,Politik
4881,saat ini warga kampung bayam seharusnya bisa h...,Politik
4946,Percayalah orang pertama yang bakal didepak Pr...,Politik
4953,Anies kehilangan Otoritas bicara etika pada de...,Politik


# Feature Engineering

## Keyword Feature

In [ ]:
kw_df = pd.read_excel(f"{DATA_DIR}/keywords_prob.xlsx")
kw_df.head()

                   keyword  Demografi  Ekonomi  Geografi  Ideologi  \
0        pendidikan rendah        1.0      0.0       0.0       0.0   
1           kelompok etnis        1.0      0.0       0.0       0.0   
2  masyarakat lintas etnis        1.0      0.0       0.0       0.0   
3             kaum difabel        1.0      0.0       0.0       0.0   
4                    bugis        1.0      0.0       0.0       0.0   

   Pertahanan dan Keamanan  Sosial Budaya  Sumber Daya Alam  Politik  
0                      0.0            0.0               0.0      0.0  
1                      0.0            0.0               0.0      0.0  
2                      0.0            0.0               0.0      0.0  
3                      0.0            0.0               0.0      0.0  
4                      0.0            0.0               0.0      0.0  

In [ ]:
train_kw_df = train_df.copy()
train_kw_df.head()

                                                text             label  \
0  kunjungan prabowo ini untuk meresmikan dan men...  Sumber Daya Alam   
1  anies dapat tepuk tangan meriah saat jadi rekt...           Politik   
2  emng bener sih pendukung 01 ada yang goblok be...         Demografi   
3  sewaktu anies bersikap kritis ke kinerja pak p...           Politik   
4  anies baswedan harap aparatur sipil negara ter...           Politik   

                                              text_2  
0  kunjung prabowo ini untuk resmi dan serah proy...  
1  anies dapat tepuk tangan riah saat jadi rektor...  
2  emng bener sih dukung 01 ada yang goblok begit...  
3  waktu anies sikap kritis ke kerja pak prabowo ...  
4  anies baswedan harap aparatur sipil negara mas...  

In [ ]:
for kw in kw_df['keyword']:
    train_kw_df[kw] = train_kw_df['text'].str.lower().str.contains(kw).astype(int)

train_kw_df.head()

<ipython-input-13-1e0113f6a834>:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df2[kw] = train_df2['text'].str.lower().str.contains(kw).astype(int)
<ipython-input-13-1e0113f6a834>:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df2[kw] = train_df2['text'].str.lower().str.contains(kw).astype(int)
<ipython-input-13-1e0113f6a834>:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once usi

                                                text             label  \
0  kunjungan prabowo ini untuk meresmikan dan men...  Sumber Daya Alam   
1  anies dapat tepuk tangan meriah saat jadi rekt...           Politik   
2  emng bener sih pendukung 01 ada yang goblok be...         Demografi   
3  sewaktu anies bersikap kritis ke kinerja pak p...           Politik   
4  anies baswedan harap aparatur sipil negara ter...           Politik   

                                              text_2  pendidikan rendah  \
0  kunjung prabowo ini untuk resmi dan serah proy...                  0   
1  anies dapat tepuk tangan riah saat jadi rektor...                  0   
2  emng bener sih dukung 01 ada yang goblok begit...                  1   
3  waktu anies sikap kritis ke kerja pak prabowo ...                  0   
4  anies baswedan harap aparatur sipil negara mas...                  0   

   kelompok etnis  masyarakat lintas etnis  kaum difabel  bugis  \
0               0                    

In [ ]:
# give me rows where all the columns is 0
train_kw_df[(train_kw_df[kw_df['keyword']] == 0).all(axis=1)].label.value_counts()

label
Politik                    53
Sumber Daya Alam            5
Sosial Budaya               4
Ideologi                    3
Pertahanan dan Keamanan     2
Ekonomi                     2
Geografi                    1
Name: count, dtype: int64

In [ ]:
# drop rows where all the columns is 0
train_kw_df.drop(train_kw_df[(train_kw_df[kw_df['keyword']] == 0).all(axis=1)].index, inplace=True)
train_kw_df.shape

(4169, 1767)

In [ ]:
train_kw_df.label.value_counts()

label
Politik                    2815
Sosial Budaya               380
Ekonomi                     264
Pertahanan dan Keamanan     257
Ideologi                    252
Sumber Daya Alam            128
Demografi                    55
Geografi                     18
Name: count, dtype: int64

# Modelling

In [ ]:
def labelvsall(label, target):
  if label == target:
    return label
  else:
    return 'Other'

In [ ]:
train_kw_df['label_pol'] = train_kw_df['label'].apply(labelvsall, target='Politik')
train_kw_df['label_sos'] = train_kw_df['label'].apply(labelvsall, target='Sosial Budaya')
train_kw_df['label_eko'] = train_kw_df['label'].apply(labelvsall, target='Ekonomi')
train_kw_df['label_aman'] = train_kw_df['label'].apply(labelvsall, target='Pertahanan dan Keamanan')
train_kw_df['label_ide'] = train_kw_df['label'].apply(labelvsall, target='Ideologi')
train_kw_df['label_sda'] = train_kw_df['label'].apply(labelvsall, target='Sumber Daya Alam')
train_kw_df['label_demo'] = train_kw_df['label'].apply(labelvsall, target='Demografi')
train_kw_df['label_geo'] = train_kw_df['label'].apply(labelvsall, target='Geografi')

<ipython-input-18-db9d6c63f39f>:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df2['label_pol'] = train_df2['label'].apply(labelvsall, target='Politik')
<ipython-input-18-db9d6c63f39f>:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df2['label_sos'] = train_df2['label'].apply(labelvsall, target='Sosial Budaya')
<ipython-input-18-db9d6c63f39f>:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all col

In [ ]:
# Split the DataFrame into training and validation sets
train_df_split, val_df_split = train_test_split(train_kw_df, test_size=0.2, stratify=train_kw_df['label'], random_state=55)

In [ ]:
train_df_split.reset_index(drop=True, inplace=True)
val_df_split.reset_index(drop=True, inplace=True)

def get_pool(train_data, valid_data, label):
  text_feat = ["text_2"]
  # embed_feat = ["ft_embed"]

  train_pool = Pool(
      data=train_data[kw_df["keyword"].to_list() + text_feat].to_numpy(),
      label=train_data[label].to_numpy(),
      text_features=text_feat,
      # embedding_features = ['ft_embed'],
      feature_names=kw_df["keyword"].to_list() + text_feat
  )

  val_pool = Pool(
      data=valid_data[kw_df["keyword"].to_list() + text_feat].to_numpy(),
      label=valid_data[label].to_numpy(),
      text_features=text_feat,
      # embedding_features = ['ft_embed'],
      feature_names=kw_df["keyword"].to_list() + text_feat
  )

  return train_pool, val_pool

def count_weight(dataset, label_col):
  class_weights_dict = dict()
  n, num_class = dataset.shape[0], len(dataset[label_col].unique())
  for label in dataset[label_col].unique():
      n_l = dataset[dataset[label_col] == label].shape[0]
      class_weights_dict[label] = n / (num_class * n_l)

  return class_weights_dict

## Main

In [ ]:
label = "label"
train_pool, val_pool = get_pool(train_df_split, val_df_split, label)
weight = count_weight(train_df_split, label)
print(weight)

{'Politik': 0.1851132326820604, 'Geografi': 29.776785714285715, 'Ekonomi': 1.9757109004739337, 'Pertahanan dan Keamanan': 2.0236650485436893, 'Ideologi': 2.063737623762376, 'Sosial Budaya': 1.371299342105263, 'Demografi': 9.474431818181818, 'Sumber Daya Alam': 4.0870098039215685}


In [ ]:
cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',

    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42,
    class_weights = weight,
    learning_rate = 0.015
)

cb_model.fit(
    train_pool,
    # eval_set=val_pool
)

0:	learn: 0.3630572	total: 29.9ms	remaining: 29.9s
100:	learn: 0.5846646	total: 1.38s	remaining: 12.3s
200:	learn: 0.6361154	total: 2.54s	remaining: 10.1s
300:	learn: 0.6571840	total: 3.66s	remaining: 8.5s
400:	learn: 0.6711335	total: 4.78s	remaining: 7.15s
500:	learn: 0.6917398	total: 6.99s	remaining: 6.96s
600:	learn: 0.7030144	total: 9.88s	remaining: 6.56s
700:	learn: 0.7092756	total: 11s	remaining: 4.69s
800:	learn: 0.7150861	total: 12.1s	remaining: 3.01s
900:	learn: 0.7215627	total: 13.2s	remaining: 1.45s
999:	learn: 0.7282166	total: 14.3s	remaining: 0us


In [ ]:
predictions = cb_model.predict(val_pool)

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_df_split[label], predictions)
print("Balanced Accuracy:", balanced_acc)

Balanced Accuracy: 0.5739217472277418


In [ ]:
predictions = cb_model.predict(val_pool)

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_df_split[label], predictions)
print("Balanced Accuracy:", balanced_acc)

Balanced Accuracy: 0.5426684156226376


## One Label vs ALL

### Geografi

In [ ]:
label = "label_geo"
train_pool, val_pool = get_pool(train_df_split, val_df_split, label)
weight = count_weight(train_df_split, label)
print(weight)

{'Other': 0.5021077988557663, 'Geografi': 119.10714285714286}


In [ ]:
cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42,
    class_weights = weight,
    learning_rate = 0.015
)

cb_model.fit(
    train_pool,
    # eval_set=val_pool
)

0:	learn: 0.8825751	total: 144ms	remaining: 2m 23s
100:	learn: 0.9948809	total: 5.32s	remaining: 47.3s
200:	learn: 0.9951821	total: 7s	remaining: 27.8s
300:	learn: 0.9954832	total: 10.4s	remaining: 24.3s
400:	learn: 0.9956338	total: 11.7s	remaining: 17.4s
500:	learn: 0.9957843	total: 15.1s	remaining: 15.1s
600:	learn: 0.9957843	total: 16s	remaining: 10.6s
700:	learn: 0.9957843	total: 16.8s	remaining: 7.15s
800:	learn: 0.9959349	total: 17.5s	remaining: 4.35s
900:	learn: 0.9960855	total: 18.3s	remaining: 2.01s
999:	learn: 0.9965371	total: 19.1s	remaining: 0us


In [ ]:
# Predict class probabilities
probabilities = cb_model.predict(val_pool, prediction_type='Probability')

class_names = cb_model.classes_  # This will get the class names
print(class_names)

result_geo = []
for instance_probs in probabilities:
    max_index = np.argmax(instance_probs)  # Index of the highest probability
    max_class = class_names[max_index]
    max_proba = instance_probs[max_index]
    result_geo.append({"class": max_class, "proba": max_proba})

['Other' 'Geografi']


### Demografi

In [ ]:
label = "label_demo"
train_pool, val_pool = get_pool(train_df_split, val_df_split, label)
weight = count_weight(train_df_split, label)
print(weight)

{'Other': 0.5066848982072318, 'Demografi': 37.89772727272727}


In [ ]:
cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42,
    class_weights = weight,
    learning_rate = 0.015
)

cb_model.fit(
    train_pool,
    # eval_set=val_pool
)

0:	learn: 0.7769642	total: 19.4ms	remaining: 19.4s
100:	learn: 0.8714127	total: 3.1s	remaining: 27.6s
200:	learn: 0.9124600	total: 7.29s	remaining: 29s
300:	learn: 0.9394285	total: 13.1s	remaining: 30.5s
400:	learn: 0.9487082	total: 17.1s	remaining: 25.5s
500:	learn: 0.9550910	total: 20.4s	remaining: 20.3s
600:	learn: 0.9564582	total: 27.8s	remaining: 18.5s
700:	learn: 0.9681232	total: 30.4s	remaining: 13s
800:	learn: 0.9690354	total: 34.2s	remaining: 8.48s
900:	learn: 0.9694914	total: 40.2s	remaining: 4.41s
999:	learn: 0.9704034	total: 42.6s	remaining: 0us


In [ ]:
# Predict class probabilities
probabilities = cb_model.predict(val_pool, prediction_type='Probability')

class_names = cb_model.classes_  # This will get the class names
print(class_names)

result_demo = []
for instance_probs in probabilities:
    max_index = np.argmax(instance_probs)  # Index of the highest probability
    max_class = class_names[max_index]
    max_proba = instance_probs[max_index]
    result_demo.append({"class": max_class, "proba": max_proba})

['Other' 'Demografi']


### Sumber Daya Alam

In [ ]:
label = "label_sda"
train_pool, val_pool = get_pool(train_df_split, val_df_split, label)
weight = count_weight(train_df_split, label)
print(weight)

{'Other': 0.5157748221466131, 'Sumber Daya Alam': 16.348039215686274}


In [ ]:
cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42,
    class_weights = weight,
    learning_rate = 0.015
)

cb_model.fit(
    train_pool,
    # eval_set=val_pool
)

0:	learn: 0.7057928	total: 15.1ms	remaining: 15.1s
100:	learn: 0.7757179	total: 977ms	remaining: 8.69s
200:	learn: 0.7823868	total: 1.79s	remaining: 7.12s
300:	learn: 0.8131675	total: 2.59s	remaining: 6.03s
400:	learn: 0.8391349	total: 3.45s	remaining: 5.15s
500:	learn: 0.8631803	total: 6.98s	remaining: 6.95s
600:	learn: 0.8818522	total: 7.93s	remaining: 5.27s
700:	learn: 0.8849682	total: 8.77s	remaining: 3.74s
800:	learn: 0.8904135	total: 9.58s	remaining: 2.38s
900:	learn: 0.8996475	total: 10.4s	remaining: 1.14s
999:	learn: 0.9128322	total: 11.2s	remaining: 0us


In [ ]:
# Predict class probabilities
probabilities = cb_model.predict(val_pool, prediction_type='Probability')

class_names = cb_model.classes_  # This will get the class names
print(class_names)

result_sda = []
for instance_probs in probabilities:
    max_index = np.argmax(instance_probs)  # Index of the highest probability
    max_class = class_names[max_index]
    max_proba = instance_probs[max_index]
    result_sda.append({"class": max_class, "proba": max_proba})

['Other' 'Sumber Daya Alam']


### Ideologi

In [ ]:
label = "label_ide"
train_pool, val_pool = get_pool(train_df_split, val_df_split, label)
weight = count_weight(train_df_split, label)
print(weight)

{'Other': 0.532237472071497, 'Ideologi': 8.254950495049505}


In [ ]:
cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42,
    class_weights = weight,
    learning_rate = 0.015
)

cb_model.fit(
    train_pool,
    # eval_set=val_pool
)

0:	learn: 0.6497193	total: 14.9ms	remaining: 14.9s
100:	learn: 0.7300281	total: 996ms	remaining: 8.86s
200:	learn: 0.7408802	total: 1.84s	remaining: 7.3s
300:	learn: 0.7526260	total: 4.28s	remaining: 9.94s
400:	learn: 0.7650700	total: 6.46s	remaining: 9.65s
500:	learn: 0.7731184	total: 7.28s	remaining: 7.25s
600:	learn: 0.8044361	total: 8.04s	remaining: 5.34s
700:	learn: 0.8309229	total: 8.85s	remaining: 3.78s
800:	learn: 0.8430631	total: 9.65s	remaining: 2.4s
900:	learn: 0.8554928	total: 10.4s	remaining: 1.15s
999:	learn: 0.8634076	total: 11.2s	remaining: 0us


In [ ]:
# Predict class probabilities
probabilities = cb_model.predict(val_pool, prediction_type='Probability')

class_names = cb_model.classes_  # This will get the class names
print(class_names)

result_ide = []
for instance_probs in probabilities:
    max_index = np.argmax(instance_probs)  # Index of the highest probability
    max_class = class_names[max_index]
    max_proba = instance_probs[max_index]
    result_ide.append({"class": max_class, "proba": max_proba})

['Other' 'Ideologi']


### Pertahanan dan Keamanan

In [ ]:
label = "label_aman"
train_pool, val_pool = get_pool(train_df_split, val_df_split, label)
weight = count_weight(train_df_split, label)
print(weight)

{'Other': 0.5329178651326303, 'Pertahanan dan Keamanan': 8.094660194174757}


In [ ]:
cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42,
    class_weights = weight,
    learning_rate = 0.015
)

cb_model.fit(
    train_pool,
    # eval_set=val_pool
)

0:	learn: 0.8366478	total: 16.1ms	remaining: 16.1s
100:	learn: 0.8469359	total: 1.05s	remaining: 9.37s
200:	learn: 0.8638329	total: 4.59s	remaining: 18.2s
300:	learn: 0.8677992	total: 5.82s	remaining: 13.5s
400:	learn: 0.8740936	total: 6.65s	remaining: 9.94s
500:	learn: 0.8790775	total: 7.49s	remaining: 7.46s
600:	learn: 0.8902177	total: 8.29s	remaining: 5.51s
700:	learn: 0.9027885	total: 9.08s	remaining: 3.87s
800:	learn: 0.9124545	total: 9.85s	remaining: 2.45s
900:	learn: 0.9187235	total: 10.7s	remaining: 1.17s
999:	learn: 0.9277490	total: 11.4s	remaining: 0us


In [ ]:
# Predict class probabilities
probabilities = cb_model.predict(val_pool, prediction_type='Probability')

class_names = cb_model.classes_  # This will get the class names
print(class_names)

result_aman = []
for instance_probs in probabilities:
    max_index = np.argmax(instance_probs)  # Index of the highest probability
    max_class = class_names[max_index]
    max_proba = instance_probs[max_index]
    result_aman.append({"class": max_class, "proba": max_proba})

['Other' 'Pertahanan dan Keamanan']


### Ekonomi

In [ ]:
label = "label_eko"
train_pool, val_pool = get_pool(train_df_split, val_df_split, label)
weight = count_weight(train_df_split, label)
print(weight)

{'Other': 0.5337708066581306, 'Ekonomi': 7.902843601895735}


In [ ]:
cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42,
    class_weights = weight,
    learning_rate = 0.015
)

cb_model.fit(
    train_pool,
    # eval_set=val_pool
)

0:	learn: 0.8418541	total: 15ms	remaining: 15s
100:	learn: 0.8461777	total: 3.62s	remaining: 32.2s
200:	learn: 0.8586567	total: 4.78s	remaining: 19s
300:	learn: 0.8624569	total: 5.59s	remaining: 13s
400:	learn: 0.8728358	total: 6.4s	remaining: 9.55s
500:	learn: 0.8790611	total: 7.18s	remaining: 7.15s
600:	learn: 0.8924505	total: 8.01s	remaining: 5.32s
700:	learn: 0.9050023	total: 8.78s	remaining: 3.75s
800:	learn: 0.9161050	total: 9.59s	remaining: 2.38s
900:	learn: 0.9325736	total: 10.4s	remaining: 1.14s
999:	learn: 0.9362235	total: 11.2s	remaining: 0us


In [ ]:
# Predict class probabilities
probabilities = cb_model.predict(val_pool, prediction_type='Probability')

class_names = cb_model.classes_  # This will get the class names
print(class_names)

result_eko = []
for instance_probs in probabilities:
    max_index = np.argmax(instance_probs)  # Index of the highest probability
    max_class = class_names[max_index]
    max_proba = instance_probs[max_index]
    result_eko.append({"class": max_class, "proba": max_proba})

['Other' 'Ekonomi']


### Sosial Budaya

In [ ]:
label = "label_sos"
train_pool, val_pool = get_pool(train_df_split, val_df_split, label)
weight = count_weight(train_df_split, label)
print(weight)

{'Other': 0.5501484658528538, 'Sosial Budaya': 5.485197368421052}


In [ ]:
cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42,
    class_weights = weight,
    learning_rate = 0.015
)

cb_model.fit(
    train_pool,
    # eval_set=val_pool
)

0:	learn: 0.7468295	total: 51.7ms	remaining: 51.6s
100:	learn: 0.7531034	total: 3.2s	remaining: 28.5s
200:	learn: 0.7598671	total: 4.04s	remaining: 16s
300:	learn: 0.7704828	total: 4.81s	remaining: 11.2s
400:	learn: 0.7776908	total: 5.6s	remaining: 8.37s
500:	learn: 0.7876944	total: 6.39s	remaining: 6.37s
600:	learn: 0.7943977	total: 7.16s	remaining: 4.75s
700:	learn: 0.7994847	total: 8.38s	remaining: 3.58s
800:	learn: 0.8039663	total: 9.21s	remaining: 2.29s
900:	learn: 0.8082781	total: 10s	remaining: 1.1s
999:	learn: 0.8137038	total: 10.8s	remaining: 0us


In [ ]:
# Predict class probabilities
probabilities = cb_model.predict(val_pool, prediction_type='Probability')

class_names = cb_model.classes_  # This will get the class names
print(class_names)

result_sos = []
for instance_probs in probabilities:
    max_index = np.argmax(instance_probs)  # Index of the highest probability
    max_class = class_names[max_index]
    max_proba = instance_probs[max_index]
    result_sos.append({"class": max_class, "proba": max_proba})

['Other' 'Sosial Budaya']


### Politik

In [ ]:
label = "label_pol"
train_pool, val_pool = get_pool(train_df_split, val_df_split, label)
weight = count_weight(train_df_split, label)
print(weight)

{'Politik': 0.7404529307282416, 'Other': 1.5397045244690675}


In [ ]:
cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42,
    class_weights = weight,
    learning_rate = 0.015
)

cb_model.fit(
    train_pool,
    # eval_set=val_pool
)

0:	learn: 0.7178462	total: 47.6ms	remaining: 47.6s
100:	learn: 0.7185016	total: 2.1s	remaining: 18.7s
200:	learn: 0.7291427	total: 2.95s	remaining: 11.7s
300:	learn: 0.7352141	total: 3.83s	remaining: 8.89s
400:	learn: 0.7376685	total: 4.62s	remaining: 6.9s
500:	learn: 0.7392229	total: 5.4s	remaining: 5.38s
600:	learn: 0.7428338	total: 6.21s	remaining: 4.12s
700:	learn: 0.7480325	total: 7.01s	remaining: 2.99s
800:	learn: 0.7511962	total: 7.81s	remaining: 1.94s
900:	learn: 0.7524927	total: 8.61s	remaining: 946ms
999:	learn: 0.7537973	total: 9.41s	remaining: 0us


In [ ]:
# Predict class probabilities
probabilities = cb_model.predict(val_pool, prediction_type='Probability')

class_names = cb_model.classes_  # This will get the class names
print(class_names)

result_pol = []
for instance_probs in probabilities:
    max_index = np.argmax(instance_probs)  # Index of the highest probability
    max_class = class_names[max_index]
    max_proba = instance_probs[max_index]
    result_pol.append({"class": max_class, "proba": max_proba})

['Politik' 'Other']


In [ ]:
for i, pred in enumerate(result_geo):
    if pred['class'] == 'Geografi':
        print(i, val_df_split['text'][i], pred['proba'])

138 dulu gue bertanya 2 kenapa tuhan tidak beri kesempatan kita menangkan prabowo sekarang gw tau jawabannya terima kasih tuhan dapitnih 0.5534862056685295
190 hadirnya anies sebagai gubernur dki waktu itu mengubah nasib warga pulau sebira adanya plts sebagai suplai listrik perbaikan sekolah air bersih hingga kapal yang lebih cepat membantu memotong waktu tempuh menjadi 2 5 jam sehingga meningkatkan kesejahteraan dan ekonomi warga azvyae 0.7421247157147091
228 anies mengajak kita memindahkan ruang palsu keruang nyata agar kita tidak terbuai mimpi 2 yang dijanjikan politisi gagasannya bisa dicubit bisa disentuh bisa dikritik bisa dinilai bahkan dengan rekam jejak tinggalkan mimpi yang penuh janji 2 hadirkan janji 2 dlm kenyataan dapitnih 0.8439971474885947
428 n rei i pak ganjar dan pak mahfud titip dipikirkan gagasannya kira apa program untuk meningkatkan sdm unggul di indonesia timur dan atau daerah kepulauan terima kasih 0.9838319092769485
440 anies seorang partai politik paham bahwa

# Other

In [ ]:
final_pred = []
for pred, pred_geo, pred_demo, pred_sda, pred_ide, pred_aman, pred_eko, pred_sos in zip(predictions, result_geo, result_demo, result_sda, result_ide, result_aman, result_eko, result_sos):
  if pred[0] == 'Base':
    final_pred.append(pred[0])
  # elif pred_geo['class'] == 'Geografi' and pred_geo['proba'] > 0.7:
  #   final_pred.append('Geografi')
  # elif pred_demo['class'] == 'Demografi' and pred_demo['proba'] > 0.5:
  #   final_pred.append('Demografi')
  # elif pred_sda['class'] == 'Sumber Daya Alam' and pred_sda['proba'] > 0.50:
  #   final_pred.append('Sumber Daya Alam')
  # elif pred_ide['class'] == 'Ideologi' and pred_ide['proba'] > 0.65:
  #   final_pred.append('Ideologi')
  # elif pred_aman['class'] == 'Pertahanan dan Keamanan' and pred_aman['proba'] > 0.5:
  #   final_pred.append('Pertahanan dan Keamanan')
  elif pred_eko['class'] == 'Ekonomi' and pred_eko['proba'] > 0.6:
    final_pred.append('Ekonomi')
  # elif pred_sos['class'] == 'Sosial Budaya' and pred_sos['proba'] > 0.58:
  #   final_pred.append('Sosial Budaya')
  else:
    final_pred.append(pred[0])

In [ ]:
# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_df_split['label'], predictions)
print("Balanced Accuracy:", balanced_acc)

Balanced Accuracy: 0.5459125523660968


In [ ]:
# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_df_split['label'], final_pred)
print("Balanced Accuracy demo:", balanced_acc)

Balanced Accuracy demo: 0.5327002559816232


In [ ]:
# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_df_split['label'], final_pred)
print("Balanced Accuracy aman:", balanced_acc)

Balanced Accuracy aman: 0.5499264136832702


In [ ]:
# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_df_split['label'], final_pred)
print("Balanced Accuracy eko:", balanced_acc)

Balanced Accuracy eko: 0.5479565282392302


In [ ]:
def get_prediction(proba=False):
  cb_model = CatBoostClassifier(
      loss_function='MultiClass',
      eval_metric='TotalF1',
      task_type="GPU",
      devices="0",
      verbose=100,
      random_state=42,
      class_weights = weight,
      learning_rate = 0.01
  )

  cb_model.fit(
      train_pool,
      # eval_set=val_pool
  )
  if proba:
    return cb_model.predict(val_pool, prediction_type='Probability'), cb_model
  else:
    return cb_model.predict(val_pool)

def format_output(prob, class_names):
    class_names = cb_model.classes_  # This will get the class names

    result_format = []
    for instance_probs in prob:
        max_index = np.argmax(instance_probs)  # Index of the highest probability
        max_class = class_names[max_index]
        max_proba = instance_probs[max_index]
        result_format.append({"class": max_class, "proba": max_proba})
    return result_format

In [ ]:
from sklearn.model_selection import StratifiedKFold

kf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

all_scores = []

train_kw_df.reset_index(drop=True, inplace=True)

for i, (train_index, test_index) in enumerate(kf.split(train_kw_df, train_kw_df['label'])):
  print(train_index[:10])
  train_df_split = train_kw_df.loc[train_index]
  val_df_split   = train_kw_df.loc[test_index]

  label = "label"
  train_pool, val_pool = get_pool(train_df_split, val_df_split, label)
  weight = count_weight(train_df_split, label)
  predictions = get_prediction()

  # label = "label_demo"
  # train_pool, val_pool = get_pool(train_df_split, val_df_split, label)
  # weight = count_weight(train_df_split, label)
  # result_demo, cb_model = get_prediction(proba=True)
  # result_demo = format_output(result_demo, cb_model)

  # label = "label_eko"
  # train_pool, val_pool = get_pool(train_df_split, val_df_split, label)
  # weight = count_weight(train_df_split, label)
  # result_eko = get_prediction()

  # label = "label_aman"
  # train_pool, val_pool = get_pool(train_df_split, val_df_split, label)
  # weight = count_weight(train_df_split, label)
  # result_aman = get_prediction()

  final_pred = []
  for pred in zip(predictions):
    if pred[0] == 'Base':
      final_pred.append(pred[0])
    # elif pred_geo['class'] == 'Geografi' and pred_geo['proba'] > 0.7:
    #   final_pred.append('Geografi')
    # elif pred_demo['class'] == 'Demografi' and pred_demo['proba'] > 0.5:
    #   final_pred.append('Demografi')
    # elif pred_sda['class'] == 'Sumber Daya Alam' and pred_sda['proba'] > 0.50:
    #   final_pred.append('Sumber Daya Alam')
    # elif pred_ide['class'] == 'Ideologi' and pred_ide['proba'] > 0.65:
    #   final_pred.append('Ideologi')
    # elif pred_aman['class'] == 'Pertahanan dan Keamanan' and pred_aman['proba'] > 0.5:
    #   final_pred.append('Pertahanan dan Keamanan')
    # elif pred_eko['class'] == 'Ekonomi' and pred_eko['proba'] > 0.6:
    #   final_pred.append('Ekonomi')
    # elif pred_sos['class'] == 'Sosial Budaya' and pred_sos['proba'] > 0.58:
    #   final_pred.append('Sosial Budaya')
    else:
      final_pred.append(pred[0])

  # Calculate the balanced accuracy
  balanced_acc = balanced_accuracy_score(val_df_split['label'], final_pred)
  all_scores.append(balanced_acc)

[0 1 2 3 4 5 6 7 8 9]
0:	learn: 0.3939574	total: 86.9ms	remaining: 1m 26s
100:	learn: 0.5839369	total: 3.37s	remaining: 30s
200:	learn: 0.6122043	total: 4.56s	remaining: 18.1s
300:	learn: 0.6360860	total: 5.71s	remaining: 13.3s
400:	learn: 0.6553662	total: 6.84s	remaining: 10.2s
500:	learn: 0.6679809	total: 7.95s	remaining: 7.92s
600:	learn: 0.6732225	total: 9.04s	remaining: 6s
700:	learn: 0.6878659	total: 10.1s	remaining: 4.33s
800:	learn: 0.6969729	total: 11.2s	remaining: 2.79s
900:	learn: 0.7056118	total: 12.3s	remaining: 1.35s
999:	learn: 0.7089357	total: 16.4s	remaining: 0us
[ 0  1  3  8 11 12 13 14 15 16]
0:	learn: 0.3879413	total: 24.3ms	remaining: 24.3s
100:	learn: 0.5806929	total: 1.39s	remaining: 12.4s
200:	learn: 0.6064820	total: 2.56s	remaining: 10.2s
300:	learn: 0.6220984	total: 3.69s	remaining: 8.57s
400:	learn: 0.6337076	total: 4.77s	remaining: 7.13s
500:	learn: 0.6414390	total: 5.88s	remaining: 5.86s
600:	learn: 0.6582586	total: 9.94s	remaining: 6.6s
700:	learn: 0.67376

In [ ]:
all_scores

[0.558956269770583,
 0.5828751886330907,
 0.4849917505994103,
 0.4968048371370719,
 0.5299628370420459]

In [ ]:
import statistics
statistics.mean(all_scores)

0.5307181766364404

In [ ]:
all_scores

[0.5918626779229836,
 0.6062327968261944,
 0.4721253736137709,
 0.5136698789515264,
 0.5303617455328259]

In [ ]:
# import statistics
print("demo : ",statistics.mean(all_scores))

demo :  0.5292088554675275


In [ ]:
print("aman : ",statistics.mean(all_scores))

aman :  0.5428504945694602


In [ ]:
old = [0.5820283225992953,
 0.6000922170615158,
 0.5244035160229987,
 0.5116900193934908,
 0.5853605551897173]

statistics.mean(old)

0.5607149260534036

In [ ]:
import statistics
statistics.mean(all_scores)  # = 20.11111111111111

0.535197771975594

In [ ]:
# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_df_split['label'], final_pred)
print("Balanced Accuracy:", balanced_acc)

Balanced Accuracy: 0.5287375543679406


In [ ]:
# Import necessary libraries
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming 'cb_model' is your trained CatBoost model and 'val_df_split' contains the validation data
# Get the class names from the CatBoost model
class_names = cb_model.classes_.tolist()
print(class_names)
class_names = ['Demografi', 'Ekonomi', 'Geografi', 'Ideologi', 'Pertahanan', 'Politik', 'SosBud', 'SDA']
# class_names = ['Other', 'Politik']

# Assuming 'val_pool' is the validation data pool and 'predictions' are made using the model
# predictions = cb_model.predict(val_pool)  # This line should be run in your environment

# Calculate balanced accuracy
balanced_acc = balanced_accuracy_score(val_df_split['label'], final_pred)
print("Balanced Accuracy:", balanced_acc)

# Generate the confusion matrix
cm = confusion_matrix(val_df_split['label'], final_pred)

# Plot the confusion matrix using seaborn
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Labels')
plt.ylabel('Actual Labels')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Geo
report = classification_report(val_df_split['label'], final_pred, target_names=class_names)
print("Classification Report:")
print(report)

In [ ]:
model_1 = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=0,
    random_state=42,
    class_weights=class_weights_dict,
    text_features=['text']
)


cv_method = RepeatedStratifiedKFold(n_splits=5, n_repeats=1, random_state=42)
cv_results_model_1 = cross_val_score(estimator=model_1, X=train_kw_df.drop("label", axis=1), y=train_kw_df["label"], cv=cv_method,
                                     verbose=100, scoring='balanced_accuracy')

print("Model 1 - Balanced Accuracy scores: ")
print(cv_results_model_1)
print("Mean: ")
print(cv_results_model_1.mean().round(3))